# Leakage and churn, with controls

Issue #9. The ladder from the ideation notebook (29 → 70 % churn with rising savings share, tenure > 1 year) could just be tenure or activity in disguise. Here the same question with the confounders held fixed: a heatmap by tenure, a logistic regression with controls, a stratified comparison, and the incremental value in the churn model.

Leakage = `cat_cash + cat_savings` share of spend, see `docs/assumptions.md` A1. `cat_savings_share` alone is the conservative variant.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

from src import style
from src.churn import RESULTS, cross_validate, feature_matrix
from src.data import load_customer_table
from src.features import TENURE_LABELS, build_customer_features

style.apply()
pd.set_option("display.width", 200)

feats = build_customer_features(load_customer_table())
train = feats[feats["split"] == "train"].reset_index(drop=True)
y = train["churned"].astype(bool).astype(int)

LEAK_BINS = [-0.01, 0, 0.05, 0.2, 0.5, 1.0]
LEAK_LABELS = ["0", "0–5 %", "5–20 %", "20–50 %", "> 50 %"]
train["leak_bin"] = pd.cut(train["leak_share"], LEAK_BINS, labels=LEAK_LABELS)
print(train.shape, "churn rate", y.mean().round(3))

## churn rate by tenure and leakage

rows are tenure buckets, columns the leakage share bins, cell = churn rate, text = n. if leakage were only tenure in disguise, the rows would be flat.

In [ ]:
rate = pd.crosstab(train["tenure_bucket"], train["leak_bin"], values=y, aggfunc="mean")
n = pd.crosstab(train["tenure_bucket"], train["leak_bin"])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(rate.to_numpy() * 100, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(LEAK_LABELS)), LEAK_LABELS)
ax.set_yticks(range(len(rate.index)), rate.index.astype(str))
for i in range(rate.shape[0]):
    for j in range(rate.shape[1]):
        if n.iat[i, j] >= 10:
            ax.text(j, i, f"{rate.iat[i, j] * 100:.0f} %\nn={n.iat[i, j]}", ha="center", va="center", fontsize=8,
                    color="white" if rate.iat[i, j] > 0.55 else style.INK)
ax.grid(False)
style.clean(ax, "churn rate by tenure and share of spend sent to other financial providers", "tenure", "leakage share")
plt.colorbar(im, ax=ax, label="% churned", shrink=0.8)
plt.tight_layout()
rate.round(2)

within almost every tenure row the `> 50 %` column is above the `0` column, and the `0–5 %` column is the lowest. so it is not flat, but it is also not a straight line: a little leakage marks an engaged customer (uses the card for everything, including a Revolut top-up now and then), a lot of leakage marks a feeder account. cells with n < 10 are blank.

## trend test, tenure > 1 year

the ladder from the ideation notebook, tested: logistic regression of churn on the ordinal leakage bin, customers active more than a year. once for `cat_savings_share` (the variable the ladder was built on) and once for `leak_share`, each time over all bins and over leakers only.

In [ ]:
old = train[train["days_active"] > 365]
y_old = old["churned"].astype(bool).astype(int)

def trend(col, frame, yy):
    bins = pd.cut(frame[col], LEAK_BINS if frame[col].min() <= 0 else LEAK_BINS[1:]).cat.codes.astype(float)
    m = sm.Logit(yy, sm.add_constant(bins)).fit(disp=0)
    return m.params.iloc[1], m.pvalues.iloc[1]

rows = []
for col in ["cat_savings_share", "leak_share"]:
    ladder = old.groupby(pd.cut(old[col], LEAK_BINS, labels=LEAK_LABELS), observed=True)["churned"].agg(lambda s: s.astype(bool).mean())
    s_all, p_all = trend(col, old, y_old)
    leakers = old[old[col] > 0]
    s_lk, p_lk = trend(col, leakers, leakers["churned"].astype(bool).astype(int))
    rows.append({"variable": col, **{f"churn {l}": round(v, 3) for l, v in ladder.items()},
                 "slope all bins": round(s_all, 3), "p all bins": f"{p_all:.1e}", "slope leakers only": round(s_lk, 3), "p leakers only": f"{p_lk:.1e}"})
pd.DataFrame(rows).set_index("variable").T

`cat_savings_share` is a clean ladder, 29 → 27 → 44 → 54 → 70 %, p < 1e-6. `leak_share` is U-shaped: customers with a little cash/savings activity churn least (18 %), then it rises to 43 %. among leakers the trend is strong for both. the zero bin of `leak_share` mixes customers who never touch these categories with one-day users, that is what bends the U.

## logistic regression with controls

`churned ~ leak_share + log(days_active) + log(n_transactions) + log(total_amount) + ecom_perc + n_country + foreign_tx_share`

`leak_share` scaled to 10-point steps so the odds ratio reads "per +10 percentage points". second model with `cat_savings_share` instead, the variant that does not depend on assumption A1.

In [ ]:
CONTROLS = ["log_days_active", "log_n_transactions", "log_total_amount", "ecom_perc", "n_country", "foreign_tx_share"]

def fit_logit(exposure):
    X = train[CONTROLS].copy()
    X.insert(0, exposure, train[exposure] * 10)
    return sm.Logit(y, sm.add_constant(X)).fit(disp=0)

models = {"leak_share": fit_logit("leak_share"), "cat_savings_share": fit_logit("cat_savings_share")}
for name, m in models.items():
    lo, hi = np.exp(m.conf_int().loc[name])
    print(f"{name:18s} OR per +10 pts = {np.exp(m.params[name]):.3f}  [{lo:.3f}, {hi:.3f}]  p = {m.pvalues[name]:.1e}")

X = sm.add_constant(train[["leak_share"] + CONTROLS])
vif = {c: variance_inflation_factor(X.to_numpy(), i) for i, c in enumerate(X.columns) if c != "const"}
print("VIF:", {k: round(v, 1) for k, v in vif.items()})
models["leak_share"].summary2().tables[1][["Coef.", "P>|z|", "[0.025", "0.975]"]].round(3)

In [ ]:
m = models["leak_share"]
params = m.params.drop("const")
ci = np.exp(m.conf_int().drop("const"))
ors = np.exp(params)

fig, ax = plt.subplots(figsize=(7, 3.8))
ypos = np.arange(len(ors))[::-1]
ax.errorbar(ors, ypos, xerr=[ors - ci[0], ci[1] - ors], fmt="o", color=style.C[0], ecolor=style.INK2, capsize=3)
ax.axvline(1, color=style.GREY, lw=1, ls="--")
ax.set_yticks(ypos, [f"{n} (per +10 pts)" if n == "leak_share" else n for n in ors.index])
ax.set_xscale("log")
ax.set_xticks([0.4, 0.6, 0.8, 1.0, 1.5, 2.0], ["0.4", "0.6", "0.8", "1", "1.5", "2"])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
style.clean(ax, "odds ratios with 95 % CI, churn ~ leakage + controls", "", "odds ratio (log scale)")
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True)
plt.tight_layout()

## stratified comparison

strata = tenure bucket × spend tercile. inside each stratum, churn rate of customers with leakage above 5 % minus customers without. pooled with stratum sizes as weights. no model, just a weighted difference.

In [ ]:
s = train[["tenure_bucket", "total_amount", "leak_share"]].copy()
s["spend_tercile"] = pd.qcut(s["total_amount"], 3, labels=["low", "mid", "high"])
s["leaker"] = s["leak_share"] > 0.05
s["y"] = y

rows = []
for (tb, sp), g in s.groupby(["tenure_bucket", "spend_tercile"], observed=True):
    a, b = g[g["leaker"]], g[~g["leaker"]]
    if len(a) >= 10 and len(b) >= 10:
        rows.append({"tenure": tb, "spend": sp, "n_leakers": len(a), "n_others": len(b),
                     "churn_leakers": a["y"].mean(), "churn_others": b["y"].mean(), "diff": a["y"].mean() - b["y"].mean()})
strata = pd.DataFrame(rows)
w = strata["n_leakers"] + strata["n_others"]
pooled = (strata["diff"] * w).sum() / w.sum()
print(f"{len(strata)} strata with ≥ 10 customers on both sides, pooled difference = {pooled * 100:+.1f} percentage points")
print(f"strata where leakers churn more: {(strata['diff'] > 0).sum()} of {len(strata)}")
strata.round(3)

## incremental value in the churn model

tuned XGBoost from `08b`, same folds, with and without the leakage columns (`cat_cash`, `cat_savings`, their shares, `leak_amount`, `leak_share`, `has_leak`).

In [ ]:
from xgboost import XGBClassifier

params = json.loads((RESULTS / "best_params.json").read_text())["xgboost"]
make = lambda: XGBClassifier(**params, random_state=0, n_jobs=-1, tree_method="hist", eval_metric="logloss")

X_full = feature_matrix(train.drop(columns="leak_bin"), "engineered")
LEAK_COLS = [c for c in X_full.columns if c.startswith(("cat_cash", "cat_savings", "leak_", "has_leak"))]
X_noleak = X_full.drop(columns=LEAK_COLS)

res = {}
for name, X_ in [("with leakage", X_full), ("without leakage", X_noleak)]:
    p, met = cross_validate(make(), X_, y.to_numpy(), n_splits=5, n_repeats=3, seed=0)
    res[name] = (p, met)
    print(f"{name:16s} AUC {met['auc'][0]:.4f} ± {met['auc'][1]:.4f}   accuracy {met['accuracy'][0]:.4f}   log loss {met['log_loss'][0]:.4f}")
print("dropped:", LEAK_COLS)

In [ ]:
import shap
from lightgbm import LGBMClassifier

lgbm_params = json.loads((RESULTS / "best_params.json").read_text())["lightgbm"]
model = LGBMClassifier(**lgbm_params, random_state=0, n_jobs=-1, verbose=-1).fit(X_full, y)
sv = shap.TreeExplainer(model).shap_values(X_full)
sv = sv[1] if isinstance(sv, list) else sv

fig, ax = plt.subplots(figsize=(7, 4))
sc = ax.scatter(X_full["leak_share"], sv[:, X_full.columns.get_loc("leak_share")], c=np.log10(X_full["days_active"]),
                cmap="viridis", s=8, alpha=0.6)
ax.axhline(0, color=style.GREY, lw=1)
style.clean(ax, "SHAP of leak_share (LightGBM), colour = log10(days_active)", "SHAP value (→ churn)", "leak_share")
ax.grid(axis="x", visible=True)
plt.colorbar(sc, ax=ax, label="log10 days active")
plt.tight_layout()

## what we can and cannot say

- customers who route money to other financial providers churn more, also at equal tenure and activity. odds ratio and stratified difference both say so, and the model gains a little AUC from the leakage columns
- the relation is not linear for `leak_share`: a little cash/savings activity goes with engaged customers, a lot with feeder accounts. for `cat_savings_share` it is a straight ladder
- no per-customer dates, so we do not know whether the money left before or after the customer stopped using the card. reverse causation (winding down = transferring the rest out) is plausible and cannot be excluded
- the leakage proxy rests on assumption A1. the `cat_savings_share` variant does not and shows the same sign
- the churn definition is unknown (A2)

wording for the slides: "customers who send money to other providers churn more, also at equal tenure", not "leakage causes churn".